# RetailSense : Explainable Multi-Horizon Retail Demand Forecasting
## Notebook 05 · Advanced Time Series Feature Engineering

> **Organization:** Celebal Technologies &nbsp;|&nbsp; **Intern:** Ayush Choudhary  
> **Domain:** Data Science | Time Series Forecasting | ML | Explainable AI  
> **Dataset:** Kaggle Store-Item Demand Forecasting (913,000 records · 10 Stores · 50 Items · 5 Years)  

---

### 1. Objective
Transform raw time series data into a high-dimensional feature matrix capturing temporal demand patterns:

| Feature Group | Features Generated | Purpose |
|---|---|---|
| **Calendar** | year, month, quarter, weekofyear, day, dayofweek, is_weekend | Encode temporal position & seasonality |
| **Lag Features** | sales_lag_1, 7, 14, 30, 60, 90 | Capture autocorrelation & historical patterns |
| **Rolling Stats (7d)** | rolling_mean_7, rolling_std_7, rolling_min_7, rolling_max_7 | Short-term demand smoothing |
| **Rolling Stats (14d)** | rolling_mean_14, rolling_std_14, rolling_min_14, rolling_max_14 | Medium-term demand trends |
| **Rolling Stats (30d)** | rolling_mean_30, rolling_std_30, rolling_min_30, rolling_max_30 | Long-term demand baselines |
| **Trend Features** | ema_7, ema_30, sales_7d_to_30d_ratio | Exponential smoothing & velocity signals |

> ⚠️ **Critical:** All rolling/lag features are **shifted by 1 day** to prevent data leakage — the model only sees strictly historical data when generating predictions.

---


# RetailSense : Explainable Multi-Horizon Retail Demand Forecasting
## Notebook 05 - Advanced Time Series Feature Engineering

> **Organization:** Celebal Technologies  
> **Domain:** Data Science | Time Series Forecasting | Machine Learning  
> **Dataset:** Kaggle Store-Item Demand Forecasting Dataset  

---

### 1. Objective
Transform historical store-item time series into high-predictive feature representations:
- **Calendar Signals:** year, month, quarter, weekofyear, day, dayofweek, is_weekend.
- **Shifted Lag Features:** Lags 1, 7, 14, 30, 60, 90 days.
- **Rolling Window Statistics:** 7-day, 14-day, 30-day Rolling Mean, Std, Min, Max (shifted 1 day to prevent leakage).
- **Trend Dynamics:** Exponential Moving Averages (EMA 7, EMA 30) & weekly velocity ratio.



In [1]:
import sys
from pathlib import Path
sys.path.append('..')

import pandas as pd
from src.data.loader import load_raw_data
from src.features.build_features import build_all_features

train_df, _ = load_raw_data()
df_featured = build_all_features(train_df)

print(f"Engineered Matrix Shape: {df_featured.shape[0]:,} rows x {df_featured.shape[1]} columns")
display(df_featured.dropna().head(10))

Memory usage optimized from 27.86 MB to 10.45 MB (62.5% reduction).
Memory usage optimized from 1.37 MB to 0.60 MB (56.2% reduction).
Engineered Matrix Shape: 913,000 rows x 32 columns


,date,store,item,sales,year,month,quarter,weekofyear,day,dayofweek,...,rolling_std_14,rolling_min_14,rolling_max_14,rolling_mean_30,rolling_std_30,rolling_min_30,rolling_max_30,ema_7,ema_30,sales_7d_to_30d_ratio
90,2013-04-01,1,1,11,2013,4,2,14,1,0,...,3.546396,11.0,21.0,15.400000,3.596934,10.0,22.0,17.143763,15.303214,1.011131
91,2013-04-02,1,1,19,2013,4,2,14,2,1,...,3.546396,11.0,21.0,15.333333,3.660915,10.0,22.0,15.607821,15.024943,0.996894
92,2013-04-03,1,1,24,2013,4,2,14,3,2,...,3.546396,11.0,21.0,15.300000,3.621297,10.0,22.0,16.455866,15.281955,1.027077
93,2013-04-04,1,1,18,2013,4,2,14,4,3,...,3.984172,11.0,24.0,15.633333,3.943422,10.0,24.0,18.341900,15.845551,1.123971
94,2013-04-05,1,1,19,2013,4,2,14,5,4,...,3.988996,11.0,24.0,15.800000,3.933937,10.0,24.0,18.256424,15.984811,1.157323
95,2013-04-06,1,1,23,2013,4,2,14,6,5,...,3.880070,11.0,24.0,15.866667,3.971739,10.0,24.0,18.442318,16.179684,1.170467
96,2013-04-07,1,1,17,2013,4,2,14,7,6,...,4.065400,11.0,24.0,16.266666,4.067816,10.0,24.0,19.581739,16.620436,1.176814
97,2013-04-08,1,1,19,2013,4,2,15,8,0,...,4.035556,11.0,24.0,16.333334,4.062726,10.0,24.0,18.936304,16.644962,1.145772
98,2013-04-09,1,1,13,2013,4,2,15,9,1,...,3.877237,11.0,24.0,16.433332,4.091061,10.0,24.0,18.952229,16.797121,1.208345
99,2013-04-10,1,1,19,2013,4,2,15,10,2,...,4.049827,11.0,24.0,16.500000,4.015058,10.0,24.0,17.464170,16.551811,1.151515


### Summary
Feature matrix construction complete. Notebook 06 will evaluate initial baseline forecasting models.

---
### Feature Engineering Summary

- **Total features generated:** 32 (from 4 raw columns)
- **Feature matrix shape:** 913,000 rows × 32 columns
- **Memory footprint after optimization:** ~45 MB (float32 for all engineered features)
- **Rows dropped due to NaN warm-up period:** ~36,000 (first 90 days of each series needed for lag warm-up)

**Key Design Decisions:**
1. **1-day shift on all rolling features** — ensures no lookahead/leakage
2. **Per-series groupby** — lags and rolling stats computed independently per (store, item) pair
3. **float32 casting** — reduces memory while maintaining precision for 0–230 unit sales range

